# Görev 3: Mini-Beyin (nn.Module ve Sınıf Yapısı)

Bir önceki görevde modelim `nn.Linear(1, 1)` tek satırdı. Bu görevde ise kendi model
sınıfımı yazıyorum: `nn.Module`'den miras alan, `MiniBrain` adında, 2 gizli katmanlı
bir MLP (Çok Katmanlı Algılayıcı).

Mimarinin akışı şöyle:

```
girdi (input_size) -> Linear -> 128 -> ReLU -> Linear -> 64 -> ReLU -> Linear -> output_size
```

Yazacağım iki metot şunlar:

- `__init__` -> modelin **hangi parçalardan** oluştuğu (üç adet `nn.Linear`)
- `forward` -> verinin bu parçalardan **hangi sırayla** geçtiği

Katman sayısı arttıkça tek tek katman tutmak dağılıyor; katmanları bir listede
saklamak, parametreleri elle toplayıp optimizer'a vermek gerekiyor. `nn.Module` bu
işi devralıyor, bu yüzden gerçek projelerde modeller hep sınıf olarak yazılıyor.
Aşağıda bunun neden böyle olduğunu adım adım gösterdim.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# her çalıştırdığımda aynı sayılar gelsin diye seed veriyorum
torch.manual_seed(42)

print(torch.__version__)

2.13.0+cu130


## 1. Adım - MiniBrain sınıfını yazıyorum

Sınıfın iki parçası var, ikisini de tek tek açıklıyorum.

### `__init__` - modelin parçalarını tanımlıyorum

**`super().__init__()`** ilk satır olmak zorunda. `nn.Module`'ün kendi kurulumu
burada çalışıyor ve modelin içine `_parameters`, `_modules` gibi boş kayıt defterleri
yerleştiriyor. Bu defterler kurulmadan katman atamaya kalkarsam hata alıyorum; birazdan
deneyip göstereceğim.

**`self.fc1 = nn.Linear(input_size, 128)`** satırı görünüşte sıradan bir atama ama
değil. `nn.Module` kendi `__setattr__`'ını yazmış durumda: bir `nn.Module` atadığımı
fark edince onu sessizce `_modules` defterine kaydediyor. Bu kayıt sayesinde daha
sonra `print(model)` mimariyi yazdırabiliyor, `model.parameters()` bütün ağırlıkları
tek seferde toplayabiliyor. Katmanları düz bir Python listesine koysaydım bu kayıt
oluşmazdı ve model o katmanları kendi parçası saymazdı.

Üç katmanın boyutları zincir gibi bağlanıyor: birinin çıkışı diğerinin girişi olmak
zorunda. `(input_size, 128)` -> `(128, 64)` -> `(64, output_size)`. Ortadaki sayılar
birbirini tutmazsa çarpım yapılamaz.

### `forward` - verinin akış sırası

`forward`, modele veri verdiğimde çalışacak olan metot. Burada yazdığım sıra, verinin
gerçek yolculuğu.

**`F.relu` neden gizli katmanlardan sonra var?** ReLU negatif değerleri sıfırlayan,
pozitifleri olduğu gibi bırakan basit bir fonksiyon (`max(0, x)`). Görevi doğrusallığı
kırmak. Aralarında aktivasyon olmasa üç `nn.Linear`'ı üst üste koymanın hiçbir anlamı
kalmazdı; doğrusal fonksiyonların bileşkesi yine doğrusal olduğu için üç katman tek
katmana eşdeğer hale gelirdi. Bunu aşağıda sayısal olarak da kanıtladım.

**Çıkış katmanından sonra neden aktivasyon yok?** `fc3`'ün ürettiği ham skorlara logit
deniyor ve kayıp fonksiyonları (`nn.CrossEntropyLoss` gibi) girdi olarak tam da bunu
bekliyor, softmax'ı kendi içinde uyguluyorlar. Buraya da aktivasyon koyarsam işlemi iki
kez yapmış olurum.

`F.relu` ile `nn.ReLU()` arasındaki fark ise şu: `F.relu` düz bir fonksiyon, `nn.ReLU()`
ise onu saran bir katman nesnesi. İkisi de aynı matematiği yapıyor; ReLU'nun öğrenilecek
parametresi olmadığı için `__init__`'te tanımlamaya gerek yok, `forward` içinde
fonksiyon olarak çağırmak yeterli.

In [2]:
class MiniBrain(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()   # nn.Module'ün kendi kurulumu, ilk satır olmalı

        # katmanlar: birinin çıkışı diğerinin girişi
        self.fc1 = nn.Linear(input_size, 128)   # girdi    -> 1. gizli katman
        self.fc2 = nn.Linear(128, 64)           # 1. gizli -> 2. gizli katman
        self.fc3 = nn.Linear(64, output_size)   # 2. gizli -> çıkış

    def forward(self, x):
        x = F.relu(self.fc1(x))   # 1. gizli katman + aktivasyon
        x = F.relu(self.fc2(x))   # 2. gizli katman + aktivasyon
        x = self.fc3(x)           # çıkış katmanı, ham skor (logit), aktivasyon yok
        return x


print("MiniBrain nn.Module'den miras alıyor mu:", issubclass(MiniBrain, nn.Module))

MiniBrain nn.Module'den miras alıyor mu: True


### `super().__init__()` gerçekten şart mı?

Lafta bırakmayıp deneyelim. Aynı sınıfı bu satır olmadan yazıp ne olduğuna bakıyorum.

In [3]:
class SuperUnutulmus(nn.Module):
    def __init__(self, input_size, output_size):
        # super().__init__()  <- bilerek yazmadım
        self.fc1 = nn.Linear(input_size, 128)


try:
    SuperUnutulmus(784, 10)
except AttributeError as e:
    print("hata:", e)

hata: cannot assign module before Module.__init__() call


Hata mesajı doğrudan söylüyor: `nn.Module.__init__()` çağrılmadan atama yapılamıyor.
Sebebi şu; katmanı kaydedecek olan `_modules` defteri henüz oluşturulmamış oluyor.

## 2. Adım - Modeli oluşturup mimariyi yazdırıyorum

Boyutları MNIST'e benzer seçtim: 28x28 bir görüntü düzleştirilince 784 sayı oluyor,
çıkışta da 10 rakam sınıfı var. Sınıfı `input_size` ve `output_size` parametreli
yazdığım için bu değerler sabit değil, aynı sınıfla başka boyutlarda model de
kurabiliyorum.

`print(model)` çağırdığımda ekrana gelen özet, `__init__` içinde kaydolan katmanların
listesi. Dikkat çeken nokta şu: bu listede `F.relu` görünmüyor, çünkü o bir katman
değil `forward` içinde çağrılan bir fonksiyon. Yani `print(model)` modelin
**parçalarını** gösteriyor, **akışını** değil. Akış sadece `forward` metodunda yazıyor.

In [4]:
input_size = 784     # 28x28 görüntü düzleştirilmiş hali
output_size = 10     # 10 sınıf

model = MiniBrain(input_size, output_size)

print(model)

MiniBrain(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
)


## 3. Adım - Modelin parametrelerine bakıyorum

`model.parameters()` çalışıyor çünkü `nn.Module` katmanları `_modules` defterinde
tutuyor ve bu deftere sırayla bakıp içlerindeki ağırlıkları topluyor. Optimizer'a tek
satırda `model.parameters()` verebilmemin sebebi bu.

Burada dikkatimi çeken bir şey var: `nn.Linear(784, 128)` katmanının ağırlık shape'i
`(784, 128)` değil `(128, 784)` çıkıyor, yani ters. Sebebi `nn.Linear`'ın içeride
şu işlemi yapması:

```
y = x @ W.T + b
```

Ağırlığı `(çıkış, giriş)` şeklinde saklayıp çarpım sırasında devriğini alıyor. Bias ise
her çıkış nöronu için bir tane, o yüzden shape'i `(çıkış,)`.

In [5]:
toplam = 0
print(f"{'parametre':<12} {'shape':<14} {'adet':>8}")
print("-" * 36)

for name, p in model.named_parameters():
    print(f"{name:<12} {str(tuple(p.shape)):<14} {p.numel():>8}")
    toplam += p.numel()

print("-" * 36)
print(f"{'TOPLAM':<12} {'':<14} {toplam:>8}")
print()
print("hepsi eğitilebilir mi:", all(p.requires_grad for p in model.parameters()))

parametre    shape              adet
------------------------------------
fc1.weight   (128, 784)       100352
fc1.bias     (128,)              128
fc2.weight   (64, 128)          8192
fc2.bias     (64,)                64
fc3.weight   (10, 64)            640
fc3.bias     (10,)                10
------------------------------------
TOPLAM                        109386

hepsi eğitilebilir mi: True


Elle de kontrol edeyim: `784 * 128 + 128 = 100480`, `128 * 64 + 64 = 8256`,
`64 * 10 + 10 = 650`. Toplam 109386. Parametrelerin neredeyse tamamı ilk katmanda,
çünkü en geniş geçiş orası.

## 4. Adım - Rastgele bir tensör üretip modele veriyorum

Mimarinin doğru kurulduğunu anlamanın en pratik yolu, beklenen boyutta bir veri
uydurup modelden geçirmek. `(batch_size, input_size)` şeklinde rastgele bir tensör
üretiyorum.

İlk boyut batch, yani aynı anda işlenen örnek sayısı. İkinci boyut ise her örneğin
özellik sayısı ve `input_size` ile birebir aynı olmak zorunda, çünkü `fc1` tam olarak
o kadar sayı bekliyor.

Modeli `model(x)` diye çağırıyorum, `model.forward(x)` diye değil. İkisinin farkına
aşağıda ayrıca değindim.

Beklentim: çıktının `(32, 10)` olması. Yani 32 örneğin her biri için 10 sınıf skoru.

In [6]:
batch_size = 32

x = torch.randn(batch_size, input_size)
print("girdi shape :", tuple(x.shape))

cikti = model(x)
print("çıktı shape :", tuple(cikti.shape))
print()
print("beklenen    :", (batch_size, output_size))
print("doğru mu    :", tuple(cikti.shape) == (batch_size, output_size))
print()
print("ilk örneğin 10 skoru:")
print(cikti[0].detach())

girdi shape : (32, 784)
çıktı shape : (32, 10)

beklenen    : (32, 10)
doğru mu    : True

ilk örneğin 10 skoru:
tensor([ 0.1314, -0.0633,  0.1735,  0.1124,  0.1599, -0.1359, -0.0092,  0.1712,
        -0.0037, -0.0520])


Boyut hatası almadan `(32, 10)` çıktı. Skorlar hem eksi hem artı; normal, çünkü model
henüz eğitilmedi ve çıkışta aktivasyon yok.

## 5. Adım - Verinin katman katman shape değişimi

Yukarıda sadece girişi ve çıkışı gördüm, aradaki katmanlarda ne olduğu gizli kaldı.
Merak ettiğim için katmanları elle, sırayla çağırıp her adımda shape yazdırıyorum.
`forward`'ın yaptığı işi açarak tekrarlıyorum yani.

Burada `torch.no_grad()` kullanıyorum çünkü sadece boyutlara bakıyorum, öğrenme yok;
gereksiz yere hesap grafiği kurulmasın.

Beklediğim akış:

```
(32, 784) -> fc1 -> (32, 128) -> relu -> (32, 128) -> fc2 -> (32, 64) -> relu -> (32, 64) -> fc3 -> (32, 10)
```

ReLU'nun shape'i değiştirmediğine dikkat: eleman eleman çalışıyor, sadece değerleri
kırpıyor. Boyutu değiştiren tek şey `nn.Linear` katmanları.

In [7]:
with torch.no_grad():
    a0 = x
    print("girdi          :", tuple(a0.shape))

    a1 = model.fc1(a0)
    print("fc1 sonrası    :", tuple(a1.shape))

    a2 = F.relu(a1)
    print("relu sonrası   :", tuple(a2.shape), " (shape aynı, değerler değişti)")

    a3 = model.fc2(a2)
    print("fc2 sonrası    :", tuple(a3.shape))

    a4 = F.relu(a3)
    print("relu sonrası   :", tuple(a4.shape))

    a5 = model.fc3(a4)
    print("fc3 sonrası    :", tuple(a5.shape))

print()
print("forward ile aynı sonuç mu:", torch.allclose(a5, cikti))

girdi          : (32, 784)
fc1 sonrası    : (32, 128)
relu sonrası   : (32, 128)  (shape aynı, değerler değişti)
fc2 sonrası    : (32, 64)
relu sonrası   : (32, 64)
fc3 sonrası    : (32, 10)

forward ile aynı sonuç mu: True


Elle yürüttüğüm akış `forward`'ın çıktısıyla birebir aynı. Bir de ReLU'nun değerlere
ne yaptığına bakayım: negatiflerin gerçekten sıfırlandığını görmek istiyorum.

In [8]:
print("relu ÖNCESİ  min: {:.4f}   negatif eleman sayısı: {}".format(
    a1.min().item(), (a1 < 0).sum().item()))
print("relu SONRASI min: {:.4f}   negatif eleman sayısı: {}".format(
    a2.min().item(), (a2 < 0).sum().item()))
print()
print("toplam eleman   :", a1.numel())

relu ÖNCESİ  min: -2.1746   negatif eleman sayısı: 1995
relu SONRASI min: 0.0000   negatif eleman sayısı: 0

toplam eleman   : 4096


## 6. Adım - Batch boyutu değişince ne oluyor?

Modeli kurarken hiçbir yerde batch boyutundan bahsetmedim. `nn.Linear` sadece son
boyuta bakıyor, önündeki eksenleri umursamıyor. Yani aynı modele istediğim kadar
örnek verebilirim. Bunu birkaç farklı batch ile deneyip doğruluyorum.

In [9]:
for bs in [1, 5, 64, 128]:
    ornek = torch.randn(bs, input_size)
    with torch.no_grad():
        out = model(ornek)
    print(f"batch_size={bs:<4} girdi {str(tuple(ornek.shape)):<12} -> çıktı {tuple(out.shape)}")

batch_size=1    girdi (1, 784)     -> çıktı (1, 10)
batch_size=5    girdi (5, 784)     -> çıktı (5, 10)
batch_size=64   girdi (64, 784)    -> çıktı (64, 10)
batch_size=128  girdi (128, 784)   -> çıktı (128, 10)


Batch değişse de çıkışın ikinci boyutu hep 10 kalıyor. Model "örnek başına ne
üreteceğini" biliyor, kaç örnek geldiği onu ilgilendirmiyor.

## 7. Adım - Yanlış boyutta veri verirsem

"Boyut hatası almadan çalıştı" demenin bir anlamı olması için, yanlış boyutta ne
olduğunu da görmem lazım. `input_size` 784 iken 500 özellikli veri veriyorum.

In [10]:
yanlis = torch.randn(32, 500)

try:
    model(yanlis)
except RuntimeError as e:
    print("hata:", e)

hata: mat1 and mat2 shapes cannot be multiplied (32x500 and 784x128)


Hata `fc1` katmanından geliyor: `(32, 500)` ile `(784, 128)` çarpılamıyor. PyTorch
hatayı tam olarak nerede patladığını söyleyerek veriyor, bu yüzden boyut hataları
genelde kolay bulunuyor.

## 8. Adım - ReLU olmasaydı gerçekten fark eder miydi?

Yukarıda "aktivasyon olmasa üç katman tek katmana eşdeğer olurdu" dedim. Bunu
denemeden geçmek istemedim.

Doğrusal bir katman `x @ W.T + b` yapıyor. Üç tanesini arka arkaya aktivasyonsuz
uygularsam, sonuç yine `x @ (tek bir matris) + (tek bir vektör)` şeklinde yazılabiliyor.
Aşağıda modelin kendi ağırlıklarını kullanarak bu tek matrisi ve tek biası elle
hesaplayıp, aktivasyonsuz akışın sonucuyla karşılaştırıyorum.

Eğer ikisi aynı çıkarsa, üç katmanlı aktivasyonsuz ağın `nn.Linear(784, 10)` tek
katmanından hiçbir farkı yok demektir. 128 ve 64 nöronluk gizli katmanlar boşa gitmiş
olur.

In [11]:
with torch.no_grad():
    # aktivasyonsuz akış: sadece üç linear
    relusuz = model.fc3(model.fc2(model.fc1(x)))

    # üç katmanı tek bir matris + tek bir bias haline getiriyorum
    W = model.fc3.weight @ model.fc2.weight @ model.fc1.weight
    b = model.fc3.weight @ (model.fc2.weight @ model.fc1.bias + model.fc2.bias) + model.fc3.bias
    tek_katman = x @ W.T + b

print("birleştirilmiş matris shape:", tuple(W.shape), " -> yani nn.Linear(784, 10)")
print()
print("aktivasyonsuz 3 katman == tek katman :", torch.allclose(relusuz, tek_katman, atol=1e-4))
print("ReLU'lu model            == tek katman :", torch.allclose(cikti, tek_katman, atol=1e-4))

birleştirilmiş matris shape: (10, 784)  -> yani nn.Linear(784, 10)

aktivasyonsuz 3 katman == tek katman : True
ReLU'lu model            == tek katman : False


Sonuç net: aktivasyonsuz üç katman, tek bir `nn.Linear(784, 10)` ile matematiksel
olarak aynı şey. ReLU devreye girince eşitlik bozuluyor, çünkü artık araya doğrusal
olmayan bir şey giriyor ve model düz bir çizgiyle anlatılamayacak ilişkileri de
öğrenebiliyor. Derinliğin işe yaramasının şartı bu.

## 9. Adım - `model(x)` mi `model.forward(x)` mi?

`forward` metodunu ben yazdım ama hiçbir yerde `model.forward(x)` diye çağırmadım,
hep `model(x)` yazdım. İkisi de aynı sonucu üretiyor, o zaman fark ne?

`nn.Module` içinde bir `__call__` metodu var. `model(x)` yazdığımda önce o çalışıyor,
o da sırasıyla kayıtlı ön-kancaları (hook), sonra benim `forward`'ımı, sonra art-kancaları
çağırıyor. `model.forward(x)` dersem bu sarmalayıcıyı atlayıp doğrudan içeri giriyorum
ve kancalar çalışmıyor.

Kancalar; ara katman çıktılarını yakalama, görselleştirme, quantization ve profil
alma gibi işlerde kullanılıyor. Bugün kanca kullanmasam bile alışkanlık olarak
`model(x)` yazmam lazım, yoksa ileride bu araçlar sessizce çalışmaz.

Aşağıda modele bir kanca takıp iki çağrım arasındaki farkı gösteriyorum.

In [12]:
kayit = []

def kanca(modul, girdi, cikti):
    kayit.append(tuple(cikti.shape))

h = model.register_forward_hook(kanca)

with torch.no_grad():
    _ = model(x)
    print("model(x) sonrası kanca kaydı        :", kayit)

    _ = model.forward(x)
    print("model.forward(x) sonrası kanca kaydı:", kayit, " (değişmedi)")

h.remove()

model(x) sonrası kanca kaydı        : [(32, 10)]
model.forward(x) sonrası kanca kaydı: [(32, 10)]  (değişmedi)


---

## Değerlendirme - forward metodunun kritik önemi

**`forward` modelin tarifidir, `__init__` sadece malzeme listesidir.**
`__init__` içinde üç `nn.Linear` tanımlamak, elimde üç katman olduğunu söylemekten
ibaret. O katmanların hangi sırayla kullanılacağı, aralarına ne konacağı, bir çıktının
başka bir yere de dallanıp dallanmayacağı `__init__`'te yazmıyor. Aynı üç katmanla
tamamen farklı mimariler kurulabilir. Modeli belirleyen şey `forward`.

**Hesap grafiği `forward` çalışırken kuruluyor.** PyTorch'un "define-by-run" yaklaşımı
tam olarak bu: mimari önceden derlenmiş sabit bir şema değil, `forward` her
çalıştığında satır satır yürütülürken oluşuyor. Hangi tensör hangi işlemden geçti, hepsi
o anda kaydediliyor. `loss.backward()` çağırdığımda geriye doğru yürünen yol işte bu
kayıt. Yani `forward` yazılmadan geriye yayılım diye bir şey de olmuyor; bir önceki
görevde elle yazdığım eğitim döngüsünün ikinci adımı zaten buydu.

**Grafik her çağrıda yeniden kurulduğu için `forward` içinde normal Python
yazabiliyorum.** `if`, döngü, hatta girdiye göre değişen bir akış koyabilirim; her
seferinde o anki yol kaydedilir. Mimarinin sabit bir dosyada tarif edilmesi gerekseydi
bu mümkün olmazdı.

**`forward`'ı ben çağırmıyorum, PyTorch çağırıyor.** `model(x)` yazdığımda araya
`nn.Module.__call__` giriyor ve kancaları çalıştırıyor. Bu yüzden `forward` benim
kodumla PyTorch'un altyapısı arasındaki sözleşme gibi; ben doldururum, çerçeve çağırır.

**Boyut uyumsuzlukları da burada ortaya çıkıyor.** `__init__` yanlış boyutlarla
kurulmuş olsa bile hiçbir hata vermiyor, model sorunsuz oluşuyor. Hata ancak veri
`forward`'tan geçmeye çalışırken patlıyor. Bu yüzden bir mimariyi yazdıktan sonra
rastgele bir tensör üretip modelden geçirmek, bu görevde yaptığım gibi, ilk yapılması
gereken kontrol.

Kısaca `__init__` neyim var sorusunun, `forward` ne yapıyorum sorusunun cevabı. Model
ikincisidir.

## Sonuç

`nn.Module`'den miras alan `MiniBrain` sınıfını yazdım; `__init__` içinde
`(784, 128)`, `(128, 64)` ve `(64, 10)` boyutlarında üç `nn.Linear` katmanı tanımladım,
`forward` içinde veriyi bu katmanlardan geçirip gizli katman aralarına `F.relu` koydum.

`print(model)` ile mimari özetini aldım, modelin 109386 eğitilebilir parametresi
olduğunu gördüm. `(32, 784)` boyutunda rastgele bir tensör üretip modele verdim, boyut
hatası almadan `(32, 10)` çıktı aldım. Aradaki shape akışının
`(32, 784) -> (32, 128) -> (32, 64) -> (32, 10)` şeklinde ilerlediğini katman katman
yazdırarak doğruladım.

Yan tarafta öğrendiklerim: `super().__init__()` olmadan katman atanamıyor, katmanları
`self`'e atamak onları otomatik olarak modele kaydediyor, aktivasyon olmadan üç katman
tek katmana eşdeğer kalıyor ve `model(x)` ile `model.forward(x)` aynı sonucu verse de
ikincisi kancaları atlıyor.

---
---

# Ekstra: BatchNorm1d ve Dropout eklemek

Ana çözüm yukarıda bitti. Bu bölüm isteğe bağlı ek; aynı mimariye katman aralarına
`nn.BatchNorm1d` ve `nn.Dropout` ekleyip ne değiştiğine bakıyorum.

**`nn.BatchNorm1d(128)`** batch içindeki örneklere bakarak her özelliği normalize
ediyor; o özelliğin batch boyunca ortalamasını çıkarıp standart sapmasına bölüyor,
sonra öğrenilebilir bir ölçek ve kaydırma uyguluyor. Amacı katmandan katmana geçerken
değerlerin dağılımının kaymasını engellemek, böylece eğitim daha hızlı ve daha kararlı
oluyor. Parametresindeki 128, o katmandaki özellik sayısı.

**`nn.Dropout(0.3)`** eğitim sırasında nöronların rastgele %30'unu geçici olarak
sıfırlıyor. Model her turda biraz farklı bir ağ görüyor, tek bir nörona bağımlı
kalamıyor. Ezberlemeyi (overfitting) zorlaştırmak için var.

**Sıralama** neden `Linear -> BatchNorm -> ReLU -> Dropout`? BatchNorm'u ReLU'dan önce
koyuyorum ki normalleştirme ham skorlara uygulansın ve ReLU dengeli dağılmış bir
girdiyle çalışsın. Dropout'u ise en sona koyuyorum; amacı zaten aktivasyondan çıkan
sinyali seyreltmek, öncesine koyarsam normalizasyon istatistiklerini de bozuyor.

İkisi de sadece gizli katman aralarına giriyor. Çıkış katmanından sonra ne BatchNorm
ne Dropout var; oradan çıkan ham skorlara dokunmak istemiyorum.

In [13]:
class MiniBrainV2(nn.Module):
    def __init__(self, input_size, output_size, dropout_p=0.3):
        super().__init__()

        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)

        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)

        self.fc3 = nn.Linear(64, output_size)

        self.dropout = nn.Dropout(dropout_p)   # aynı nesneyi iki yerde kullanabilirim

    def forward(self, x):
        # 1. gizli katman: linear -> batchnorm -> relu -> dropout
        x = self.dropout(F.relu(self.bn1(self.fc1(x))))

        # 2. gizli katman: aynı sıra
        x = self.dropout(F.relu(self.bn2(self.fc2(x))))

        # çıkış: ham skor
        x = self.fc3(x)
        return x


torch.manual_seed(42)
model_v2 = MiniBrainV2(input_size, output_size)

print(model_v2)

MiniBrainV2(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


Özette artık BatchNorm ve Dropout katmanları da görünüyor, çünkü ikisi de birer
`nn.Module` ve `__init__` içinde `self`'e atandılar. `F.relu` yine görünmüyor.

Dropout'u tek bir nesne olarak tanımlayıp iki yerde kullanabildim, çünkü Dropout'un
saklayacağı ağırlığı yok. BatchNorm için aynısını yapamazdım; her BatchNorm katmanı
kendi ölçek/kaydırma parametrelerini ve kendi çalışan istatistiklerini tutuyor, o
yüzden ayrı ayrı tanımlamak zorundayım.

Şimdi parametre sayısındaki değişime ve shape akışına bakıyorum.

In [14]:
toplam_v1 = sum(p.numel() for p in model.parameters())
toplam_v2 = sum(p.numel() for p in model_v2.parameters())

print("MiniBrain   parametre sayısı:", toplam_v1)
print("MiniBrainV2 parametre sayısı:", toplam_v2)
print("fark                        :", toplam_v2 - toplam_v1, " (BatchNorm'un ölçek + kaydırma değerleri)")
print()

model_v2.eval()   # shape'e bakarken dropout kapalı olsun
with torch.no_grad():
    print("girdi        :", tuple(x.shape))
    h = model_v2.fc1(x);            print("fc1 sonrası  :", tuple(h.shape))
    h = model_v2.bn1(h);            print("bn1 sonrası  :", tuple(h.shape), " (shape aynı, dağılım değişti)")
    h = F.relu(h);                  print("relu sonrası :", tuple(h.shape))
    h = model_v2.dropout(h);        print("dropout sonr.:", tuple(h.shape))
    h = model_v2.fc2(h);            print("fc2 sonrası  :", tuple(h.shape))
    h = model_v2.bn2(h);            print("bn2 sonrası  :", tuple(h.shape))
    h = F.relu(h)
    h = model_v2.dropout(h)
    h = model_v2.fc3(h);            print("fc3 sonrası  :", tuple(h.shape))

MiniBrain   parametre sayısı: 109386
MiniBrainV2 parametre sayısı: 109770
fark                        : 384  (BatchNorm'un ölçek + kaydırma değerleri)

girdi        : (32, 784)
fc1 sonrası  : (32, 128)
bn1 sonrası  : (32, 128)  (shape aynı, dağılım değişti)
relu sonrası : (32, 128)
dropout sonr.: (32, 128)
fc2 sonrası  : (32, 64)
bn2 sonrası  : (32, 64)
fc3 sonrası  : (32, 10)


Shape akışı ana modelle birebir aynı: `(32, 784) -> (32, 128) -> (32, 64) -> (32, 10)`.
BatchNorm ve Dropout boyut değiştirmiyor, sadece değerlere müdahale ediyor. Fark
parametre sayısında: BatchNorm katmanları `2 * 128 + 2 * 64 = 384` yeni öğrenilebilir
değer getirdi.

Bir de BatchNorm'un değerlere ne yaptığına bakayım. Normalize ettiğini iddia ediyorum,
o zaman çıktının ortalaması 0'a, standart sapması 1'e yakın olmalı.

In [15]:
model_v2.train()   # BatchNorm'un batch istatistiğini kullanması için train modu
with torch.no_grad():
    once = model_v2.fc1(x)
    sonra = model_v2.bn1(once)

print("bn ÖNCESİ  ortalama: {:7.4f}   std: {:6.4f}".format(once.mean().item(), once.std().item()))
print("bn SONRASI ortalama: {:7.4f}   std: {:6.4f}".format(sonra.mean().item(), sonra.std().item()))

bn ÖNCESİ  ortalama:  0.0003   std: 0.5800
bn SONRASI ortalama: -0.0000   std: 1.0001


### `model.train()` ve `model.eval()` neden şart oldu?

Ana modelde bu iki metodu hiç kullanmadım, çünkü `nn.Linear` eğitimde de tahminde de
aynı işi yapıyor. Ama BatchNorm ve Dropout iki modda farklı davranıyor:

- **Dropout:** `train()` modunda nöron söndürüyor, `eval()` modunda hiçbir şey yapmıyor.
- **BatchNorm:** `train()` modunda o anki batch'in istatistiğini kullanıyor ve aynı
  zamanda uzun vadeli bir ortalama biriktiriyor; `eval()` modunda ise biriktirdiği bu
  ortalamayı kullanıyor. Böylece tahmin sonucu, aynı batch'te başka hangi örneklerin
  bulunduğuna bağlı olmuyor.

Bu yüzden bu katmanlar devreye girince eğitim öncesi `model.train()`, değerlendirme
öncesi `model.eval()` çağırmak zorunlu hale geliyor. Unutulursa model hata vermiyor,
sadece sessizce yanlış sonuç üretiyor. Aşağıda aynı girdiyi iki kez verip farkı
gösteriyorum.

In [16]:
model_v2.train()
with torch.no_grad():
    t1 = model_v2(x)
    t2 = model_v2(x)

model_v2.eval()
with torch.no_grad():
    e1 = model_v2(x)
    e2 = model_v2(x)

print("train() modunda iki çağrı aynı mı:", torch.allclose(t1, t2), " (dropout rastgele söndürüyor)")
print("eval()  modunda iki çağrı aynı mı:", torch.allclose(e1, e2), " (dropout kapalı, sonuç sabit)")
print()
print("train ile eval çıktısı aynı mı   :", torch.allclose(t1, e1))
print("ilk örnek train :", t1[0][:4])
print("ilk örnek eval  :", e1[0][:4])

train() modunda iki çağrı aynı mı:

 False  (dropout rastgele söndürüyor)
eval()  modunda iki çağrı aynı mı: True  (dropout kapalı, sonuç sabit)

train ile eval çıktısı aynı mı   : False
ilk örnek train : tensor([-0.9994, -0.5559,  0.6994, -0.4653])
ilk örnek eval  : tensor([-0.2081, -0.1565, -0.0076, -0.0207])


Sonuçlar beklediğim gibi: `train()` modunda aynı girdi iki farklı çıktı veriyor,
`eval()` modunda çıktı sabitleniyor.

Bir de BatchNorm'un bilinen kısıtını deneyeyim. `train()` modunda tek örnekli bir batch
verirsem, o özelliğin batch boyunca standart sapması hesaplanamıyor.

In [17]:
tek_ornek = torch.randn(1, input_size)

model_v2.train()
try:
    model_v2(tek_ornek)
except ValueError as e:
    print("train modunda tek örnek -> hata:", e)

model_v2.eval()
with torch.no_grad():
    print()
    print("eval modunda tek örnek -> çıktı shape:", tuple(model_v2(tek_ornek).shape))

train modunda tek örnek -> hata: Expected more than 1 value per channel when training, got input size torch.Size([1, 128])

eval modunda tek örnek -> çıktı shape: (1, 10)


### Ekstra bölümün özeti

Aynı mimariye `nn.BatchNorm1d` ve `nn.Dropout` ekledim, sıralamayı
`Linear -> BatchNorm -> ReLU -> Dropout` seçtim. Shape akışı değişmedi, sadece 384
parametre eklendi ve modele iki yeni davranış girdi: değerlerin katman aralarında
normalize edilmesi ve eğitim sırasında rastgele nöron söndürme.

Asıl öğrendiğim şey bu iki katmanın modu önemsemesi oldu. `nn.Linear`'lı sade modelde
`train()` / `eval()` çağırmasam da bir şey değişmiyordu; bunlar girince çağırmak zorunlu
hale geldi, çünkü ikisi de eğitimde ve tahminde farklı iş yapıyor.